# 04 — Final checks (interview type, run inventory)

Reads the per-person prediction files that Notebook 03 already saved. **No training, no GPU,
runs in about a minute.** Put this notebook in the same folder as the `outputs` folder.

1. **Inventory**: every 3b run found under `outputs/`, its participant count and official-split result.
2. **Interview-type check**: E-DAIC mixes human-operated Wizard-of-Oz interviews (IDs < 600) with
   autonomous-agent interviews (IDs ≥ 600), and the official test set is agent-only. If depression
   rates or recording conditions differ between the two, a face-embedding model could be detecting
   *interview type* rather than depression. We check (a) prevalence by type, (b) AUC within each
   type, (c) whether the model adds anything over interview type alone.

In [ ]:
import os, glob, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from scipy.stats import fisher_exact
ROOT = "outputs"
rng = np.random.default_rng(42)

def boot_auc(y, p, n=2000):
    y, p = np.asarray(y), np.asarray(p); vals = []
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    if len(pos) < 2 or len(neg) < 2: return np.nan, np.nan, np.nan
    for _ in range(n):
        i = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]
        vals.append(roc_auc_score(y[i], p[i]))
    return roc_auc_score(y, p), *np.percentile(vals, [2.5, 97.5])
fmt = lambda a, l, h: "n/a" if np.isnan(a) else f"{a:.3f} [{l:.3f}, {h:.3f}]"
print("ready")

## 1. Inventory of saved 3b runs

In [ ]:
OOF = sorted(glob.glob(os.path.join(ROOT, "**", "*_oof_predictions.csv"), recursive=True))
OFFS = sorted(glob.glob(os.path.join(ROOT, "**", "*_official_split.csv"), recursive=True))
print("cross-validation prediction files:")
for f in OOF:
    d = pd.read_csv(f); models = [c[2:] for c in d.columns if c.startswith("p_")]
    print(f"  {f}\n     n = {len(d)} | depressed {int(d.phq_binary.sum())} | models: {', '.join(models)}")
print("\nofficial-split result files:")
for f in OFFS:
    print(f"  {f}"); print(pd.read_csv(f).to_string(index=False)); print()

## 2. Interview-type check

Set `RUNS` to the prediction files you want checked (the inventory above lists them). By default
every "cnn subset" file is checked.

In [ ]:
RUNS = [f for f in OOF if "cnn subset" in f]
SPLIT_ID = 600            # IDs below = Wizard-of-Oz (human-operated), at or above = autonomous agent

for f in RUNS:
    d = pd.read_csv(f); d["agent"] = (d.pid >= SPLIT_ID).astype(int)
    y = d.phq_binary.values; folder = os.path.basename(os.path.dirname(f))
    print("=" * 100); print(f"{folder}  ({os.path.basename(f)}, n = {len(d)})")
    tab = pd.crosstab(d.agent.map({0: "WoZ (<600)", 1: "agent (>=600)"}), d.phq_binary.map({0: "not dep", 1: "dep"}))
    tab["% dep"] = (100 * tab["dep"] / tab.sum(axis=1)).round(1)
    _, pf = fisher_exact(pd.crosstab(d.agent, d.phq_binary).values)
    print(tab.to_string()); print(f"Fisher exact p (prevalence differs by interview type) = {pf:.3f}")
    a_t = roc_auc_score(y, d.agent)
    print(f"interview type alone as a predictor: AUC {max(a_t, 1 - a_t):.3f} (0.5 = no information)")

    rows = []
    for c in [c for c in d.columns if c.startswith("p_")]:
        r = dict(model=c[2:].replace("_", " [", 1) + "]", overall=fmt(*boot_auc(y, d[c])))
        for g, name in [(0, "WoZ (<600)"), (1, "agent (>=600)")]:
            m = d.agent.values == g; r[name] = fmt(*boot_auc(y[m], d[c].values[m]))
        rows.append(r)
    print("\nAUC [95% CI] overall and within each interview type:")
    print(pd.DataFrame(rows).to_string(index=False))

    # does the model add information beyond interview type? (cross-validated stacking, as for gender)
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    lr = LogisticRegression(max_iter=1000)
    base = cross_val_predict(lr, d[["agent"]].values, y, cv=skf, method="predict_proba")[:, 1]
    for c in [c for c in d.columns if c.startswith("p_") and c.endswith("_cnn")]:
        z = np.log(np.clip(d[c], 1e-6, 1 - 1e-6) / np.clip(1 - d[c], 1e-6, 1))
        both = cross_val_predict(lr, np.c_[d.agent, z], y, cv=skf, method="predict_proba")[:, 1]
        print(f"stacked: interview type only AUC {roc_auc_score(y, base):.3f} | "
              f"type + {c[2:]} AUC {roc_auc_score(y, both):.3f}")
    print()

## What to send back

Paste the whole output of sections 1 and 2. Reading it:

| Pattern | Meaning |
|---|---|
| prevalence similar (Fisher p > 0.05), within-type AUCs close to overall | interview type is **not** the explanation |
| prevalence differs **and** within-type AUCs ≈ 0.5 | the embedding model mostly detects interview type: a confound |
| type + model ≫ type alone | the model carries information beyond interview type |